# S7-02: 에이전트 패턴 — Agent Loop, ReAct, Environment Inspection
**Skilljar L05-L07: Agents and Tools / Environment Inspection / Workflows vs Agents**

## 학습 목표
- Tool Use 기반 에이전트 루프를 직접 구현한다
- ReAct (Thought → Action → Observation) 패턴을 이해하고 적용한다
- 환경 관찰(Environment Inspection)을 통한 에이전트 적응을 구현한다
- Guardrails (반복 제한, 인간 승인, 도구 권한)를 에이전트에 적용한다

## 사전 준비
1. `.env` 파일에 API 키 설정:
```
ANTHROPIC_API_KEY="YOUR_API_KEY_HERE"
```

In [ ]:
# 패키지 설치
%pip install anthropic python-dotenv

In [ ]:
# 환경 설정
import json
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

print("환경 설정 완료")

---
## Exercise 1: 기본 에이전트 루프 — 계산기 에이전트

### 문제
Claude가 **도구를 자율적으로 선택하여 사용**하는 기본 에이전트 루프를 구현하세요.

**도구 3개:**
- `add(a, b)`: 두 수의 합
- `multiply(a, b)`: 두 수의 곱
- `power(base, exponent)`: 거듭제곱

**에이전트 루프 구조:**
```python
while response.stop_reason == "tool_use":
    # 1. tool_use 블록에서 도구명/입력 추출
    # 2. 도구 실행
    # 3. tool_result를 messages에 추가
    # 4. 다시 Claude 호출
```

### 테스트 쿼리
```
(3 + 5) * 2의 4제곱을 계산해줘
```
예상 도구 호출 순서: `add(3, 5)` → `power(2, 4)` → `multiply(8, 16)` → 최종 답: 128

### 기대 출력
```
[Turn 1] Tool: add({"a": 3, "b": 5}) → 8
[Turn 2] Tool: power({"base": 2, "exponent": 4}) → 16
[Turn 3] Tool: multiply({"a": 8, "b": 16}) → 128
[Turn 4] 최종 답변: (3 + 5) * 2^4 = 128
```

In [ ]:
# TODO: 기본 에이전트 루프를 구현하세요

# 1. 도구 정의 (tools 리스트)
# YOUR CODE HERE

# 2. 도구 실행 함수
# YOUR CODE HERE

# 3. 에이전트 루프
# YOUR CODE HERE

In [ ]:
# === 솔루션 ===

# 1. 도구 정의
tools = [
    {
        "name": "add",
        "description": "두 수를 더한다.",
        "input_schema": {
            "type": "object",
            "properties": {
                "a": {"type": "number", "description": "첫 번째 수"},
                "b": {"type": "number", "description": "두 번째 수"}
            },
            "required": ["a", "b"]
        }
    },
    {
        "name": "multiply",
        "description": "두 수를 곱한다.",
        "input_schema": {
            "type": "object",
            "properties": {
                "a": {"type": "number", "description": "첫 번째 수"},
                "b": {"type": "number", "description": "두 번째 수"}
            },
            "required": ["a", "b"]
        }
    },
    {
        "name": "power",
        "description": "거듭제곱을 계산한다.",
        "input_schema": {
            "type": "object",
            "properties": {
                "base": {"type": "number", "description": "밑"},
                "exponent": {"type": "number", "description": "지수"}
            },
            "required": ["base", "exponent"]
        }
    }
]

# 2. 도구 실행 함수
def execute_tool(name: str, inputs: dict) -> str:
    if name == "add":
        result = inputs["a"] + inputs["b"]
    elif name == "multiply":
        result = inputs["a"] * inputs["b"]
    elif name == "power":
        result = inputs["base"] ** inputs["exponent"]
    else:
        return json.dumps({"error": f"Unknown tool: {name}"})
    return json.dumps({"result": result})

# 3. 에이전트 루프
def calculator_agent(query: str, max_turns: int = 10) -> str:
    messages = [{"role": "user", "content": query}]
    turn = 0

    while turn < max_turns:
        turn += 1
        response = client.messages.create(
            model=model, max_tokens=1000,
            tools=tools, messages=messages
        )

        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            final = "".join(b.text for b in response.content if hasattr(b, "text"))
            print(f"[Turn {turn}] 최종 답변: {final[:200]}")
            return final

        # tool_use 처리
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = execute_tool(block.name, block.input)
                print(f"[Turn {turn}] Tool: {block.name}({json.dumps(block.input)}) → {result}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result
                })

        messages.append({"role": "user", "content": tool_results})

    return "최대 반복 횟수 도달"

# 실행
agent_result = calculator_agent("(3 + 5) * 2의 4제곱을 계산해줘")
print(f"\n결과: {agent_result}")

In [ ]:
# 검증 함수
def verify_exercise_1():
    """Exercise 1 결과를 검증한다."""
    checks = []

    # Check 1: agent_result 존재
    checks.append('agent_result' in globals() and agent_result is not None)

    # Check 2: 결과에 128이 포함되어 있는지
    if 'agent_result' in globals() and agent_result:
        checks.append('128' in str(agent_result))
    else:
        checks.append(False)

    # Check 3: tools 리스트가 3개의 도구를 포함하는지
    if 'tools' in globals():
        checks.append(len(tools) == 3)
    else:
        checks.append(False)

    # Check 4: execute_tool 함수가 올바르게 동작하는지
    if 'execute_tool' in globals():
        try:
            r1 = json.loads(execute_tool("add", {"a": 3, "b": 5}))
            r2 = json.loads(execute_tool("multiply", {"a": 8, "b": 16}))
            checks.append(r1["result"] == 8 and r2["result"] == 128)
        except Exception:
            checks.append(False)
    else:
        checks.append(False)

    passed = sum(checks)
    total = len(checks)
    print(f"검증 결과: {passed}/{total} 통과")
    for i, c in enumerate(checks, 1):
        print(f"  Check {i}: {'PASS' if c else 'FAIL'}")
    return passed == total

verify_exercise_1()

---
## Exercise 2: ReAct 패턴 — 정보 수집 에이전트

### 문제
ReAct (Reasoning + Acting) 패턴을 명시적으로 구현하세요.
에이전트가 각 단계에서 **Thought → Action → Observation**을 기록하도록 합니다.

**도구 2개:**
- `search_materials(material_type)`: 건축 재료 정보를 검색한다 (시뮬레이션)
- `compare_costs(material_a, material_b)`: 두 재료의 비용을 비교한다 (시뮬레이션)

**시뮬레이션 데이터:**
```python
materials_db = {
    "concrete": {"strength": "24-60 MPa", "cost_per_m3": 150000, "fire_resistance": "excellent"},
    "steel": {"strength": "235-460 MPa", "cost_per_ton": 1200000, "fire_resistance": "poor"},
    "wood": {"strength": "5-20 MPa", "cost_per_m3": 300000, "fire_resistance": "poor"},
    "masonry": {"strength": "5-15 MPa", "cost_per_m3": 100000, "fire_resistance": "good"}
}
```

### 테스트 쿼리
```
5층 건물에 적합한 구조 재료를 추천해줘. 콘크리트와 철골을 비교해줘.
```

### 기대 출력
```
[Thought] 콘크리트와 철골 정보를 먼저 조회해야 한다
[Action] search_materials("concrete")
[Observation] {"strength": "24-60 MPa", ...}
[Thought] 이제 철골 정보도 조회하자
[Action] search_materials("steel")
[Observation] {"strength": "235-460 MPa", ...}
[Thought] 두 재료를 비교하자
[Action] compare_costs("concrete", "steel")
[Observation] {"comparison": ...}
[Final] 추천: ...
```

In [ ]:
# TODO: ReAct 패턴 에이전트를 구현하세요

materials_db = {
    "concrete": {"strength": "24-60 MPa", "cost_per_m3": 150000, "fire_resistance": "excellent"},
    "steel": {"strength": "235-460 MPa", "cost_per_ton": 1200000, "fire_resistance": "poor"},
    "wood": {"strength": "5-20 MPa", "cost_per_m3": 300000, "fire_resistance": "poor"},
    "masonry": {"strength": "5-15 MPa", "cost_per_m3": 100000, "fire_resistance": "good"}
}

# 도구 정의
# YOUR CODE HERE

# 도구 실행 함수
# YOUR CODE HERE

# ReAct 에이전트 루프
# YOUR CODE HERE

In [ ]:
# === 솔루션 ===

materials_db = {
    "concrete": {"strength": "24-60 MPa", "cost_per_m3": 150000, "fire_resistance": "excellent"},
    "steel": {"strength": "235-460 MPa", "cost_per_ton": 1200000, "fire_resistance": "poor"},
    "wood": {"strength": "5-20 MPa", "cost_per_m3": 300000, "fire_resistance": "poor"},
    "masonry": {"strength": "5-15 MPa", "cost_per_m3": 100000, "fire_resistance": "good"}
}

react_tools = [
    {
        "name": "search_materials",
        "description": "건축 재료 정보를 검색한다. 지원 재료: concrete, steel, wood, masonry",
        "input_schema": {
            "type": "object",
            "properties": {
                "material_type": {"type": "string", "description": "재료 유형"}
            },
            "required": ["material_type"]
        }
    },
    {
        "name": "compare_costs",
        "description": "두 건축 재료의 비용을 비교한다.",
        "input_schema": {
            "type": "object",
            "properties": {
                "material_a": {"type": "string", "description": "첫 번째 재료"},
                "material_b": {"type": "string", "description": "두 번째 재료"}
            },
            "required": ["material_a", "material_b"]
        }
    }
]

def execute_react_tool(name: str, inputs: dict) -> str:
    if name == "search_materials":
        material = inputs["material_type"].lower()
        if material in materials_db:
            return json.dumps(materials_db[material], ensure_ascii=False)
        return json.dumps({"error": f"'{material}' 재료를 찾을 수 없습니다"})
    elif name == "compare_costs":
        a = inputs["material_a"].lower()
        b = inputs["material_b"].lower()
        if a in materials_db and b in materials_db:
            return json.dumps({
                a: materials_db[a],
                b: materials_db[b],
                "recommendation": f"{a}가 일반적으로 경제적" if "cost_per_m3" in materials_db[a] and materials_db[a].get("cost_per_m3", float('inf')) < materials_db[b].get("cost_per_m3", float('inf')) else f"비교에 추가 정보가 필요"
            }, ensure_ascii=False)
        return json.dumps({"error": "재료를 찾을 수 없습니다"})
    return json.dumps({"error": f"Unknown tool: {name}"})

def react_agent(query: str, max_turns: int = 8) -> str:
    """ReAct 패턴 에이전트. Thought/Action/Observation을 추적한다."""
    system = (
        "당신은 건축 재료 전문 에이전트입니다. "
        "각 단계에서 먼저 현재 상황을 분석(Thought)하고, "
        "필요한 도구를 사용(Action)하세요. "
        "모든 정보를 수집한 후 최종 추천을 제시하세요."
    )
    messages = [{"role": "user", "content": query}]
    trace = []  # ReAct 추적 로그

    for turn in range(max_turns):
        response = client.messages.create(
            model=model, max_tokens=1500,
            system=system, tools=react_tools, messages=messages
        )

        messages.append({"role": "assistant", "content": response.content})

        # Thought 추출 (텍스트 블록)
        for block in response.content:
            if hasattr(block, "text") and block.text.strip():
                thought = block.text.strip()[:150]
                trace.append(f"[Thought] {thought}")
                print(f"[Thought] {thought}")

        if response.stop_reason == "end_turn":
            final = "".join(b.text for b in response.content if hasattr(b, "text"))
            trace.append(f"[Final] {final[:200]}")
            print(f"[Final] 최종 답변 생성 완료")
            return final

        # Action + Observation
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                action = f"{block.name}({json.dumps(block.input, ensure_ascii=False)})"
                trace.append(f"[Action] {action}")
                print(f"[Action] {action}")

                result = execute_react_tool(block.name, block.input)
                trace.append(f"[Observation] {result[:100]}")
                print(f"[Observation] {result[:100]}")

                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result
                })

        messages.append({"role": "user", "content": tool_results})

    return "최대 반복 횟수 도달"

# 실행
react_result = react_agent(
    "5층 건물에 적합한 구조 재료를 추천해줘. 콘크리트와 철골을 비교해줘."
)
print(f"\n=== 최종 추천 ===")
print(react_result)

In [ ]:
# 검증 함수
def verify_exercise_2():
    """Exercise 2 결과를 검증한다."""
    checks = []

    # Check 1: react_result 존재
    checks.append('react_result' in globals() and react_result is not None)

    # Check 2: 결과에 콘크리트/철골 관련 내용이 있는지
    if 'react_result' in globals() and react_result:
        has_content = any(term in react_result for term in ["콘크리트", "철골", "concrete", "steel"])
        checks.append(has_content)
    else:
        checks.append(False)

    # Check 3: react_tools가 2개인지
    if 'react_tools' in globals():
        checks.append(len(react_tools) == 2)
    else:
        checks.append(False)

    # Check 4: execute_react_tool이 올바르게 동작하는지
    if 'execute_react_tool' in globals():
        try:
            r = json.loads(execute_react_tool("search_materials", {"material_type": "concrete"}))
            checks.append("strength" in r)
        except Exception:
            checks.append(False)
    else:
        checks.append(False)

    passed = sum(checks)
    total = len(checks)
    print(f"검증 결과: {passed}/{total} 통과")
    for i, c in enumerate(checks, 1):
        print(f"  Check {i}: {'PASS' if c else 'FAIL'}")
    return passed == total

verify_exercise_2()

---
## Exercise 3: Guardrails — 안전한 에이전트

### 문제
다음 3가지 Guardrails를 포함한 에이전트를 구현하세요:

1. **반복 제한 (Max Iterations)**: 최대 5회 도구 호출
2. **도구 권한 제어 (Tool Permissions)**: `allowed_tools` 목록에 없는 도구는 거부
3. **인간 승인 (Human Approval)**: `dangerous_tools` 목록의 도구는 실행 전 확인

**도구 4개:**
- `read_data(source)`: 데이터 읽기 (안전)
- `analyze(data)`: 데이터 분석 (안전)
- `write_report(content)`: 보고서 쓰기 (승인 필요)
- `delete_data(source)`: 데이터 삭제 (차단)

### 기대 동작
```
read_data → 실행
analyze → 실행
write_report → "[승인 요청] 실행을 허가하시겠습니까? (y/n)"
delete_data → "[차단] 'delete_data'는 허용되지 않은 도구입니다."
```

In [ ]:
# TODO: Guardrails가 포함된 에이전트를 구현하세요

# allowed_tools = ["read_data", "analyze", "write_report"]
# dangerous_tools = ["write_report"]

# YOUR CODE HERE

In [ ]:
# === 솔루션 ===

guardrail_tools = [
    {
        "name": "read_data",
        "description": "지정된 소스에서 데이터를 읽는다.",
        "input_schema": {
            "type": "object",
            "properties": {
                "source": {"type": "string", "description": "데이터 소스명"}
            },
            "required": ["source"]
        }
    },
    {
        "name": "analyze",
        "description": "데이터를 분석한다.",
        "input_schema": {
            "type": "object",
            "properties": {
                "data": {"type": "string", "description": "분석할 데이터"}
            },
            "required": ["data"]
        }
    },
    {
        "name": "write_report",
        "description": "보고서를 작성한다.",
        "input_schema": {
            "type": "object",
            "properties": {
                "content": {"type": "string", "description": "보고서 내용"}
            },
            "required": ["content"]
        }
    },
    {
        "name": "delete_data",
        "description": "데이터를 삭제한다.",
        "input_schema": {
            "type": "object",
            "properties": {
                "source": {"type": "string", "description": "삭제할 데이터 소스"}
            },
            "required": ["source"]
        }
    }
]

def execute_guardrail_tool(name: str, inputs: dict) -> str:
    if name == "read_data":
        return json.dumps({"data": f"{inputs['source']}의 구조 데이터: 기둥 20개, 보 30개, 슬래브 10개"})
    elif name == "analyze":
        return json.dumps({"analysis": f"분석 결과: 데이터 패턴 식별 완료. 주요 발견 3건."})
    elif name == "write_report":
        return json.dumps({"status": "보고서 작성 완료", "path": "/reports/output.md"})
    elif name == "delete_data":
        return json.dumps({"status": "삭제 완료"})
    return json.dumps({"error": "Unknown tool"})

def safe_agent(
    query: str,
    max_iterations: int = 5,
    allowed_tools: list[str] = None,
    dangerous_tools: list[str] = None,
    auto_approve: bool = True  # 노트북에서는 자동 승인
) -> str:
    """Guardrails가 포함된 안전한 에이전트."""
    messages = [{"role": "user", "content": query}]
    guardrail_log = []

    for iteration in range(max_iterations):
        response = client.messages.create(
            model=model, max_tokens=1500,
            tools=guardrail_tools, messages=messages
        )

        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            final = "".join(b.text for b in response.content if hasattr(b, "text"))
            return final

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                tool_name = block.name

                # Guardrail 1: 도구 권한 확인
                if allowed_tools and tool_name not in allowed_tools:
                    msg = f"[차단] '{tool_name}'은 허용되지 않은 도구입니다."
                    print(msg)
                    guardrail_log.append(msg)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps({"error": msg}),
                        "is_error": True
                    })
                    continue

                # Guardrail 2: 위험 도구 승인
                if dangerous_tools and tool_name in dangerous_tools:
                    msg = f"[승인 요청] '{tool_name}' 실행을 허가하시겠습니까?"
                    print(msg)
                    guardrail_log.append(msg)
                    if not auto_approve:
                        approval = input("  (y/n): ").strip().lower()
                        if approval != "y":
                            tool_results.append({
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": json.dumps({"error": "사용자가 거부했습니다"}),
                                "is_error": True
                            })
                            continue
                    else:
                        print("  → 자동 승인")

                # 도구 실행
                result = execute_guardrail_tool(tool_name, block.input)
                print(f"[실행] {tool_name} → {result[:80]}")
                guardrail_log.append(f"[실행] {tool_name}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result
                })

        messages.append({"role": "user", "content": tool_results})

    # Guardrail 3: 최대 반복 도달
    msg = f"[제한] 최대 반복 횟수 {max_iterations}회에 도달"
    print(msg)
    guardrail_log.append(msg)
    return msg

# 실행
print("=== 안전한 에이전트 실행 ===")
safe_result = safe_agent(
    "건물 구조 데이터를 읽고, 분석하고, 보고서를 작성해줘. 그리고 원본 데이터를 삭제해줘.",
    max_iterations=5,
    allowed_tools=["read_data", "analyze", "write_report"],
    dangerous_tools=["write_report"],
    auto_approve=True
)
print(f"\n=== 결과 ===")
print(safe_result[:300])

In [ ]:
# 검증 함수
def verify_exercise_3():
    """Exercise 3 결과를 검증한다."""
    checks = []

    # Check 1: safe_result 존재
    checks.append('safe_result' in globals() and safe_result is not None)

    # Check 2: safe_agent 함수가 정의되었는지
    checks.append('safe_agent' in globals() and callable(safe_agent))

    # Check 3: guardrail_tools가 4개인지
    if 'guardrail_tools' in globals():
        checks.append(len(guardrail_tools) == 4)
    else:
        checks.append(False)

    # Check 4: execute_guardrail_tool이 올바르게 동작하는지
    if 'execute_guardrail_tool' in globals():
        try:
            r = json.loads(execute_guardrail_tool("read_data", {"source": "test"}))
            checks.append("data" in r)
        except Exception:
            checks.append(False)
    else:
        checks.append(False)

    passed = sum(checks)
    total = len(checks)
    print(f"검증 결과: {passed}/{total} 통과")
    for i, c in enumerate(checks, 1):
        print(f"  Check {i}: {'PASS' if c else 'FAIL'}")
    return passed == total

verify_exercise_3()

---
## Exercise 4: 건축공학 응용 — 구조 재료 추천 에이전트

### 문제
건축 프로젝트 정보를 입력받아 **자율적으로 재료를 조사하고 추천**하는 에이전트를 구현하세요.

**도구 3개:**
- `get_project_requirements(project_type)`: 프로젝트 유형별 구조 요구사항 조회
- `get_material_properties(material)`: 재료 특성 조회
- `evaluate_suitability(material, requirements)`: 재료-요구사항 적합성 평가

**에이전트가 스스로 결정할 사항:**
- 어떤 재료를 조사할지
- 몇 개의 재료를 비교할지
- 어떤 순서로 평가할지

### 테스트 쿼리
```
지하 2층, 지상 15층 오피스 건물의 주요 구조 재료를 추천해줘.
내진설계범주 D, 공기 단축이 중요한 상황이야.
```

In [ ]:
# TODO: 구조 재료 추천 에이전트를 구현하세요
# YOUR CODE HERE

In [ ]:
# === 솔루션 ===

# 시뮬레이션 데이터
project_requirements_db = {
    "office_highrise": {
        "min_concrete_strength": 40,
        "seismic_category": "D",
        "fire_rating_hours": 3,
        "lateral_system": "moment frame or shear wall",
        "key_concerns": ["drift control", "vibration", "column-free space"]
    },
    "residential_midrise": {
        "min_concrete_strength": 27,
        "seismic_category": "C",
        "fire_rating_hours": 2,
        "lateral_system": "bearing wall",
        "key_concerns": ["cost", "sound insulation", "partition flexibility"]
    }
}

material_properties_db = {
    "rc_frame": {
        "name": "RC 라멘 구조",
        "strength_range": "24-60 MPa",
        "construction_speed": "slow",
        "seismic_performance": "good with ductile detailing",
        "cost_index": 1.0,
        "fire_resistance": "excellent"
    },
    "steel_frame": {
        "name": "철골 구조",
        "strength_range": "235-460 MPa",
        "construction_speed": "fast",
        "seismic_performance": "excellent",
        "cost_index": 1.3,
        "fire_resistance": "poor (requires fireproofing)"
    },
    "src": {
        "name": "SRC 구조 (철골철근콘크리트)",
        "strength_range": "combined high",
        "construction_speed": "medium",
        "seismic_performance": "excellent",
        "cost_index": 1.5,
        "fire_resistance": "good"
    }
}

struct_agent_tools = [
    {
        "name": "get_project_requirements",
        "description": "프로젝트 유형별 구조 요구사항을 조회한다. 유형: office_highrise, residential_midrise",
        "input_schema": {
            "type": "object",
            "properties": {
                "project_type": {"type": "string", "description": "프로젝트 유형"}
            },
            "required": ["project_type"]
        }
    },
    {
        "name": "get_material_properties",
        "description": "구조 재료의 특성을 조회한다. 재료: rc_frame, steel_frame, src",
        "input_schema": {
            "type": "object",
            "properties": {
                "material": {"type": "string", "description": "재료 유형"}
            },
            "required": ["material"]
        }
    },
    {
        "name": "evaluate_suitability",
        "description": "재료와 프로젝트 요구사항의 적합성을 평가한다.",
        "input_schema": {
            "type": "object",
            "properties": {
                "material": {"type": "string", "description": "재료 유형"},
                "project_type": {"type": "string", "description": "프로젝트 유형"}
            },
            "required": ["material", "project_type"]
        }
    }
]

def execute_struct_tool(name: str, inputs: dict) -> str:
    if name == "get_project_requirements":
        pt = inputs["project_type"]
        if pt in project_requirements_db:
            return json.dumps(project_requirements_db[pt], ensure_ascii=False)
        return json.dumps({"error": f"프로젝트 유형 '{pt}'를 찾을 수 없습니다"})
    elif name == "get_material_properties":
        mat = inputs["material"]
        if mat in material_properties_db:
            return json.dumps(material_properties_db[mat], ensure_ascii=False)
        return json.dumps({"error": f"재료 '{mat}'를 찾을 수 없습니다"})
    elif name == "evaluate_suitability":
        mat = inputs["material"]
        pt = inputs["project_type"]
        if mat in material_properties_db and pt in project_requirements_db:
            props = material_properties_db[mat]
            reqs = project_requirements_db[pt]
            score = 70  # 기본 점수
            if props["seismic_performance"] == "excellent":
                score += 15
            if props["construction_speed"] == "fast":
                score += 10
            if props["fire_resistance"] in ["excellent", "good"]:
                score += 5
            return json.dumps({
                "material": mat, "project": pt,
                "suitability_score": score,
                "strengths": [k for k, v in props.items() if v in ["excellent", "fast"]],
                "concerns": [k for k, v in props.items() if v in ["poor", "slow"]]
            }, ensure_ascii=False)
        return json.dumps({"error": "평가할 수 없습니다"})
    return json.dumps({"error": f"Unknown tool: {name}"})

def structural_material_agent(query: str, max_turns: int = 10) -> str:
    system = (
        "당신은 건축 구조 재료 전문 에이전트입니다. "
        "프로젝트 요구사항을 파악하고, 후보 재료를 조사하고, 적합성을 평가하여 "
        "최적의 재료를 추천합니다. 반드시 도구를 사용하여 데이터 기반 추천을 하세요."
    )
    messages = [{"role": "user", "content": query}]

    for turn in range(max_turns):
        response = client.messages.create(
            model=model, max_tokens=2000,
            system=system, tools=struct_agent_tools, messages=messages
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            return "".join(b.text for b in response.content if hasattr(b, "text"))

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = execute_struct_tool(block.name, block.input)
                print(f"  [Tool] {block.name}({json.dumps(block.input, ensure_ascii=False)[:60]}) → {result[:80]}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result
                })
        messages.append({"role": "user", "content": tool_results})

    return "최대 반복 횟수 도달"

# 실행
print("=== 구조 재료 추천 에이전트 ===")
struct_result = structural_material_agent(
    "지하 2층, 지상 15층 오피스 건물의 주요 구조 재료를 추천해줘. "
    "내진설계범주 D, 공기 단축이 중요한 상황이야."
)
print(f"\n=== 최종 추천 ===")
print(struct_result)

In [ ]:
# 검증 함수
def verify_exercise_4():
    """Exercise 4 결과를 검증한다."""
    checks = []

    # Check 1: struct_result 존재
    checks.append('struct_result' in globals() and struct_result is not None)

    # Check 2: 결과가 충분한 길이
    if 'struct_result' in globals() and struct_result:
        checks.append(len(struct_result) > 100)
    else:
        checks.append(False)

    # Check 3: struct_agent_tools가 3개인지
    if 'struct_agent_tools' in globals():
        checks.append(len(struct_agent_tools) == 3)
    else:
        checks.append(False)

    passed = sum(checks)
    total = len(checks)
    print(f"검증 결과: {passed}/{total} 통과")
    for i, c in enumerate(checks, 1):
        print(f"  Check {i}: {'PASS' if c else 'FAIL'}")
    return passed == total

verify_exercise_4()